# 08.04 Prompt Hub — 프롬프트 버전 관리

프롬프트는 사실상 **코드**지만, 엔지니어가 아닌 사람이 자주 고치고, 고치는 주기도 짧다. 매번 재배포하지 않으려면 프롬프트를 **애플리케이션 코드와 분리된 저장소**에 두고 버전 핀을 찍는다. LangSmith Prompt Hub 가 그 역할을 한다.

## 학습 목표

- `client.push_prompt("name", object=...)` 로 프롬프트를 업로드한다
- Commit SHA 고정 vs `prod`·`staging` 태그 참조의 차이를 이해한다
- f-string vs mustache 템플릿의 변수 처리 차이를 비교한다
- Playground 에서 실험 → 커밋 → 태그 플로우를 숙지한다
- 런타임에서 `client.pull_prompt("name:prod")` 로 프롬프트를 주입해 `create_agent(system_prompt=...)` 에 연결한다
- CI 테스트에서 **특정 커밋 해시를 고정**해 회귀를 방지한다

## 사전 준비

```dotenv
LANGSMITH_API_KEY=lsv2_pt_...
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=langsmith-prompt-hub
OPENAI_API_KEY=sk-...
```

Prompt Hub 는 트레이싱과 별개지만, push/pull 호출도 트레이스에 남기므로 같은 프로젝트 이름을 쓰면 편하다. SDK 는 `langsmith >= 0.1.99` 필요.

In [ ]:
# !pip install -U langsmith langchain langchain-openai

from dotenv import load_dotenv
import os
load_dotenv(override=True)

assert os.environ.get("LANGSMITH_API_KEY"), "LANGSMITH_API_KEY 미설정"
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 미설정"
os.environ.setdefault("LANGSMITH_PROJECT", "langsmith-prompt-hub")
print("프로젝트:", os.environ["LANGSMITH_PROJECT"])

## 8.04.1 Prompt 생성 · 푸시

가장 단순한 형태는 `ChatPromptTemplate` 을 만든 뒤 `client.push_prompt("이름", object=prompt)` 로 올리는 것. 첫 push 는 새 prompt 를 생성하고, 이후 push 는 새 commit 을 쌓는다. 반환 URL 로 UI 에서 바로 열린다.

<!-- phase-c:embed -->
![Prompts 허브 목록](../assets/images/langsmith/04_prompt_hub/01_prompt_hub_list.png)

*Prompts 목록 — `city-list`(1 commit) 과 `weather-bot`(2 commits). Visibility 컬럼에 Private/Public 배지, Last Commit 컬럼에 짧은 SHA(`e3d8a720`, `054fbccb`) 표시. 상단 `+ Prompt` 버튼 외에 `+ Webhook`(프롬프트 변경 시 웹훅 호출) 옵션 노출.*

In [ ]:
from langsmith import Client
from langchain_core.prompts import ChatPromptTemplate

client = Client()

weather_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 날씨 봇입니다. 한 문장으로 간결히 답하세요."),
    ("human", "{city} 의 날씨 알려줘"),
])

url = client.push_prompt("weather-bot", object=weather_prompt)
print("push 완료 ->", url)

## 8.04.2 Commit SHA 고정 vs 태그 (`prod`, `staging`)

프롬프트는 Git 처럼 동작한다.

| 참조 방식 | 예 | 특성 | 언제 쓰나 |
|-----------|----|------|-----------|
| **Commit SHA** | `weather-bot:12344e88` | 불변, 정확히 그 버전을 핀 | CI 회귀 테스트, 재현 필요 시 |
| **Tag** | `weather-bot:prod`, `weather-bot:staging` | 이동형 — 같은 태그가 다른 커밋을 가리키도록 바꿀 수 있다 | 런타임 배포 슬롯 |
| **최신** | `weather-bot` | 가장 최근 커밋 | 개발 초기에만, 프로덕션 금지 |

태그는 애플리케이션 코드를 재배포하지 않고 프롬프트만 교체할 수 있는 핵심 장치다. UI 의 `Commits` 뷰에서 특정 커밋에 `prod` 태그를 **promote** 한다.

<!-- phase-c:embed -->
![Prompt 상세 + commit + Messages + Code Snippet](../assets/images/langsmith/04_prompt_hub/02_prompt_detail.png)

*`weather-bot` 상세. 상단 commit `054fbccb` + 탭(Messages / Code Snippet / Comments). Messages 에 System/User 템플릿(`{city}` 변수), Code Snippet 에 `client.pull_prompt("weather-bot")` Python SDK 예시. 좌측 Environments 패널의 Production/Staging 슬롯에 `Promote` 로 태깅. 우상단 Permissions/Fork/Playground 버튼.*

In [ ]:
# push 후 반환되는 URL 에서 commit hash 를 꺼내 핀을 찍어본다
commit_hash = url.split("/")[-1]  # 예: '12344e88'
print("핀으로 쓸 commit:", commit_hash)

# 같은 프롬프트를 한번 더 수정해 새 커밋을 만든다
weather_prompt_v2 = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 날씨 봇입니다. 한국어로 한 문장 답. 필요시 이모지 하나."),
    ("human", "{city} 의 날씨 알려줘"),
])
url_v2 = client.push_prompt("weather-bot", object=weather_prompt_v2)
print("v2 push ->", url_v2)
print("-> UI 에서 v1 커밋에 prod 태그, v2 커밋에 staging 태그를 붙여보자")

## 8.04.3 f-string vs mustache

템플릿 엔진 두 가지의 특성은 실무에서 자주 문제가 된다.

| 항목 | f-string (`{var}`) | mustache (`{{var}}`) |
|------|--------------------|------------------------|
| 기본값 | ✅ LangChain 기본 | 별도 선택 |
| JSON 예시 포함 | `{{` 이스케이프 필요 | 이스케이프 불필요 |
| 조건문 · 반복 | ❌ 미지원 | ✅ `{{#users}}...{{/users}}` |
| 중첩 키 | 제한적 | ✅ `{{user.name}}` |
| Playground 변수 지정 | 자동 감지 | 수동 지정 (Inputs) |

JSON/코드 예시를 많이 끼워 넣거나 반복/조건이 들어가면 mustache 가, 단순 변수 치환이면 f-string 이 편하다.

In [ ]:
# mustache 템플릿 예시 — template_format="mustache"
mustache_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "도시 목록을 한국어로 한 줄씩 소개하세요."),
        ("human", "{{#cities}}- {{name}} ({{country}})\n{{/cities}}"),
    ],
    template_format="mustache",
)

client.push_prompt("city-list", object=mustache_prompt)
# 반복 변수는 리스트로 바인딩
print(mustache_prompt.invoke({"cities": [
    {"name": "서울", "country": "KR"},
    {"name": "도쿄", "country": "JP"},
]}).to_messages()[-1].content)

## 8.04.4 Playground — 실험에서 커밋까지

UI 의 **Playground** 는 프롬프트 + 모델 + 입력 변수를 같이 돌려보는 공간이다. 흐름:

1. 프롬프트 페이지에서 `Open in Playground` 클릭
2. 모델·temperature·출력 스키마·tools 를 사이드 패널에서 조정
3. 변수 값을 넣고 `Run` — 결과와 token/cost 가 곧바로 기록됨
4. `Compare` 로 같은 입력에 대한 **여러 프롬프트/모델 출력 병렬 비교** (pairwise 의 수동 버전)
5. 만족스러운 상태에서 `Save as...` -> 새 커밋 생성, 필요하면 `prod` 태그로 promote

Playground 에서 돌린 모든 run 은 Experiments 뷰로 넘어가 `03_datasets_and_evaluation` 의 데이터셋과 직접 연결 가능하다.

<!-- phase-c:embed -->
![Playground — prompt 실험](../assets/images/langsmith/04_prompt_hub/03_playground.png)

*Playground — prompt 를 로드한 상태에서 SYSTEM/HUMAN 메시지 편집 + `{question}` 변수 입력 + Output 생성. 모델은 `gpt-5.4` 기본, + Message / + Output Schema / + Tool / f-string↔mustache 스위처 제공. `Set up Evaluation` 버튼으로 dataset 을 붙여 실험으로 확장 가능.*

## 8.04.5 런타임 주입 — `pull_prompt` -> `create_agent`

애플리케이션에서는 배포 슬롯 태그를 `pull_prompt` 로 끌어와 **LLM/에이전트에 바로 꽂는다**. 프롬프트를 고치면 재배포 없이 다음 요청부터 반영된다.

In [ ]:
from langchain_openai import ChatOpenAI

# 1) 프롬프트를 태그로 끌어온다 — 운영 시엔 :prod 태그 권장
prompt = client.pull_prompt("weather-bot")  # 실제 운영: "weather-bot:prod"

# 2) 모델과 체인으로 바로 연결
model = ChatOpenAI(model="gpt-4.1-mini")
chain = prompt | model
result = chain.invoke({"city": "서울"})
print(result.content)

In [ ]:
# 에이전트에 system_prompt 로 꽂을 때는 system 메시지 텍스트만 꺼낸다
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 반환한다."""
    return f"{city}: 맑음, 21°C"

system_text = prompt.messages[0].prompt.template  # 'system' 메시지의 원문 템플릿
agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[get_weather],
    system_prompt=system_text,      # Prompt Hub 의 최신 system 을 그대로 주입
)

out = agent.invoke({"messages": [{"role": "user", "content": "부산 날씨?"}]})
print(out["messages"][-1].content)

## 8.04.6 CI 에서 특정 커밋 해시 고정

프로덕션 배포 슬롯은 `:prod` 태그로 받되, **회귀 테스트는 반드시 commit SHA 를 핀**해야 한다. 그래야 "통과한 테스트 이후 누군가 프롬프트를 고쳤다" 는 사고가 자동으로 차단된다.

In [ ]:
# 환경변수로 핀 버전을 주입 — CI 에서 PROMPT_PIN 을 설정
PROMPT_PIN = os.environ.get("WEATHER_BOT_PIN", commit_hash)
pinned = client.pull_prompt(f"weather-bot:{PROMPT_PIN}")

# CI 테스트 예: 핀된 프롬프트가 기대 문자열을 포함하는가
rendered = pinned.invoke({"city": "서울"}).to_messages()[0].content
assert "날씨 봇" in rendered, "system prompt 가 예상과 다름 — prompt commit 확인 필요"
print(f"CI 통과 (pin: weather-bot:{PROMPT_PIN})")

### 배포 패턴 요약

| 환경 | 참조 | 이유 |
|------|------|------|
| dev / 로컬 | `weather-bot` (최신) | 즉시 반영 |
| staging | `weather-bot:staging` | 태그만 promote 해서 테스트 |
| prod | `weather-bot:prod` | 태그 이동으로 무중단 롤아웃, 롤백은 태그 되돌리기 |
| CI | `weather-bot:{SHA}` | 재현 가능, 프롬프트 변경이 바로 테스트 실패로 감지됨 |

`03_datasets_and_evaluation` 의 experiment 에 `prompt_commit` 을 metadata 로 넣으면 **어떤 커밋 기준에서 낸 수치인지** UI 에서 바로 추적된다.

## 체크리스트

- [ ] `client.push_prompt("name", object=prompt)` 로 새 커밋 생성
- [ ] UI 의 Commits 뷰에서 커밋에 `prod` / `staging` 태그 promote
- [ ] f-string vs mustache 선택 기준 이해 (반복/조건/중첩 -> mustache)
- [ ] Playground 에서 실험 -> Save -> 태그 promote 흐름 경험
- [ ] `client.pull_prompt("name:prod")` 로 런타임 주입
- [ ] CI 에서 `name:{SHA}` 핀으로 회귀 테스트

## 다음

- `05_production_monitoring.ipynb` — 배포된 프롬프트를 사용하는 프로덕션 에이전트를 대시보드·알림·PII 방어로 감싸기

## 참고

- Prompt engineering 개념: https://docs.langchain.com/langsmith/prompt-engineering-concepts
- Manage prompts programmatically: https://docs.langchain.com/langsmith/manage-prompts-programmatically
- Playground: https://docs.langchain.com/langsmith/playground